# 第 3 周练习 — 本地开源模型（CPU 最小版）

## 练习目标（理念）

第 3 周核心是 **Hugging Face**：分词器（Tokenizer）、管道（Pipeline）、以及在本地跑 **开源模型**。

Day-5 完整会议纪要项目在 Colab GPU 上使用 Whisper + Llama-8B；本笔记本是一个 **可在 CPU 上本地跑通的最小版本**：

1. 对比多个分词器如何把同一句英文切成 token  
2. 用开源摘要模型把会议逐字稿压成「会议纪要」风格短文  

## 和本课 Week 3 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| `AutoTokenizer` | 对同一 `text` 比较 gpt2 / BERT / Qwen |
| `pipeline` | `summarization` 管道一键推理 |
| 开源权重 | `Falconsai/text_summarization`（CPU 友好） |
| 与 Day-5 对照 | 文末指向 Whisper + 量化 Llama 完整版 |

## 怎么跑

1. 已安装 `transformers`（及依赖 `torch`）  
2. 从上到下 Shift+Enter；分词器对比无需 HF 登录（选用非门控模型）  
3. 首次跑摘要会从 Hub 下载权重，需网络  


In [1]:
# ========== 导入：分词器与推理管道 ==========
# AutoTokenizer：按模型名加载对应分词器（无大权重，很快）
# pipeline：高层任务 API，这里后面用于 summarization
from transformers import AutoTokenizer, pipeline


## 1. 分词器（Tokenizer）对比

同一段文本，不同模型的分词器会切出 **不同数量、不同表面形式** 的 token。

- 分词器本身很小（没有完整模型权重），所以这一格通常几秒内跑完  
- 选用的都是 **非门控（ungated）** 模型名，一般 **不需要** Hugging Face 登录即可 `from_pretrained`


In [2]:
# ========== 同一句话：三个分词器各切成什么？ ==========

# 待编码的示例英文（字符串保持原文，便于对照 token）
text = "Tokenization turns text into integers a model can process."
# 依次加载：GPT-2 BPE、BERT WordPiece、Qwen2.5 Instruct 分词器
for name in ["gpt2", "bert-base-uncased", "Qwen/Qwen2.5-0.5B-Instruct"]:
    # 按 Hub 模型名拉取分词器配置与词表
    tok = AutoTokenizer.from_pretrained(name)
    # encode：文本 → token id 列表（整数，模型真正吃的输入）
    ids = tok.encode(text)
    # 打印：模型名、token 个数、前 8 个可读 token 预览
    print(f"{name:<28} {len(ids):>3} tokens  ->  {tok.convert_ids_to_tokens(ids)[:8]} ...")


gpt2                          11 tokens  ->  ['Token', 'ization', 'Ġturns', 'Ġtext', 'Ġinto', 'Ġintegers', 'Ġa', 'Ġmodel'] ...
bert-base-uncased             13 tokens  ->  ['[CLS]', 'token', '##ization', 'turns', 'text', 'into', 'integers', 'a'] ...


Qwen/Qwen2.5-0.5B-Instruct    11 tokens  ->  ['Token', 'ization', 'Ġturns', 'Ġtext', 'Ġinto', 'Ġintegers', 'Ġa', 'Ġmodel'] ...


## 2. 用开源模型做会议纪要（Summarization）

本地 **摘要管道**（开放权重、默认可在 CPU 跑）把较长的会议逐字稿压成短摘要，模拟「会议纪要」产出。

注意：这是通用 summarization，不是 Day-5 那种带结构化字段的 LLM 纪要；完整 GPU 版见文末。


In [3]:
# ========== 会议逐字稿 → 摘要管道 → 打印「纪要」 ==========

# 多说话人会议记录（英文内容是业务数据，保持原样）
transcript = """
Alex: Welcome everyone. The goal today is to lock the launch plan for the mobile app.
Maria: Beta testing is done. We found two minor bugs, both fixed and verified yesterday.
Sam: Marketing is ready. We'll send the announcement email on Friday morning.
Alex: Great. Let's ship to the app stores Thursday so Friday's email points to a live app.
Maria: I'll submit the build Thursday and monitor the review queue.
Sam: I'll prepare the social posts and schedule them for Friday.
Alex: Perfect. We meet again Monday to review launch metrics.
"""

# 任务型管道：summarization + 指定开源摘要模型（首次会下载权重）
summarizer = pipeline("summarization", model="Falconsai/text_summarization")
# 调用管道：限制摘要长度；do_sample=False 使结果更稳定可复现
minutes = summarizer(transcript, max_length=80, min_length=25, do_sample=False)[0]["summary_text"]
# 打印标题与摘要正文（英文标签保持原样）
print("MEETING MINUTES\n" + "-" * 14 + "\n" + minutes)


Device set to use cpu


Both `max_new_tokens` (=256) and `max_length`(=80) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


MEETING MINUTES
--------------
Alex: Great. Let's ship to the app stores Thursday so Friday's email points to a live app . Sam: Marketing is ready. We'll send the announcement email on Friday morning .


**完整版（Colab GPU）提示**

若要用真实音频：先用 `openai/whisper` 转写，再用量化后的 `meta-llama/Llama-3.1-8B-Instruct` 生成**结构化**会议记录——详见课程 **第 5 天** 笔记本。
